# Room inference: detector ? crops ? shared classifier

????????????????????? quantitative room evaluation ???? expert ground truth. ??? detector baseline ?????????????????????? detector ???????????????


In [ ]:
from pathlib import Path
import json, os, sys, yaml
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from ultralytics import YOLO
from scripts.model_inference import FurniturePredictor
from scripts.detect_then_classify import detect_and_classify, export_room, file_hash


In [ ]:
config = yaml.safe_load((ROOT/'configs/inference/room-yolov8n-coco-baseline.yaml').read_text(encoding='utf-8'))
detector_weights = ROOT / config['detector']['weights']  # ????????????? detector checkpoint ?????????????
checkpoint = ROOT / 'release/checkpoint/best.pt'
if not checkpoint.is_file():
    suite = json.loads((ROOT/'artifacts/runs/suites/furniture-v2-screening.json').read_text(encoding='utf-8'))
    done = [r for r in suite['runs'] if r['status'] == 'completed']
    if not done: raise FileNotFoundError('???????? completed classifier checkpoint')
    checkpoint = Path(done[0]['attempts'][-1]['run_dir'])/'best.pt'
    print('DEMONSTRATION checkpoint, not final selection')
if not detector_weights.is_file(): raise FileNotFoundError('????????? baseline ??? README ????????? detector checkpoint')
predictor = FurniturePredictor(checkpoint, device='cpu')
detector = YOLO(str(detector_weights))


In [ ]:
image_path = ROOT/'artifacts/room-inference-demo/source.jpg'  # ????????????????????????
if not image_path.is_file(): raise FileNotFoundError('????? image_path ???????????????')
provenance = {'detector_checkpoint':str(detector_weights.resolve()), 'detector_sha256':file_hash(detector_weights),
    'classifier_checkpoint':str(checkpoint.resolve()), 'classifier_sha256':file_hash(checkpoint),
    'config':str((ROOT/'configs/inference/room-yolov8n-coco-baseline.yaml').resolve()),
    'config_sha256':file_hash(ROOT/'configs/inference/room-yolov8n-coco-baseline.yaml')}
result = detect_and_classify(image_path, detector, predictor, config['detector']['confidence'],
    class_mapping=config['detector_to_category'], padding_fraction=config['crop']['padding_fraction'],
    min_size=config['crop']['min_size'], imgsz=config['detector']['imgsz'], provenance=provenance)
result


????? object ?? bbox ?????????????? EXIF, detector label ????? category/style ??? classifier ??????; confidence ?????? compatibility score
